In [2]:
import json

union_set = set()
# valid ---
with open('folio_valid.json', 'r', encoding='utf-8') as f:
    valid_data = json.load(f)
valid_ids = set()
for d in valid_data:
    valid_ids.add(d["story_id"])
    union_set.add(d["story_id"])

# test ---
with open('folio_test.json', 'r', encoding='utf-8') as f:
    test_data = json.load(f)
test_ids = set()
for d in test_data:
    test_ids.add(d["story_id"])
    union_set.add(d["story_id"])

# print out
print("valid:", sorted(valid_ids))
print("test:", sorted(test_ids))

print("Count shared stories:", len(valid_ids) + len(test_ids) - len(union_set))

valid: [0, 2, 26, 46, 58, 64, 96, 107, 131, 140, 183, 191, 203, 213, 219, 232, 234, 245, 258, 259, 262, 282, 292, 319, 340, 350, 355, 361, 368, 385, 441, 452, 459, 467, 471, 472]
test: [20, 51, 79, 80, 83, 101, 120, 124, 151, 152, 159, 166, 184, 192, 197, 198, 217, 256, 280, 295, 306, 315, 318, 322, 330, 343, 352, 363, 379, 380, 386, 426, 435, 442, 456, 460, 483]
Count shared stories: 0


# Dependencies

In [ ]:
!python -m pip install nltk

# Fix syntax manually

Find errors

In [53]:
from nltk.sem import Expression
import re

def translate_fol(fol_text: str):
    # '-' --> '_'
    fol_text = re.sub(r'(?<=[a-zA-Z0-9])-(?=[a-zA-Z0-9])', '_', fol_text)

    replacements = {
        '∀': ' all ', 
        '∃': ' exists ',
        '∧': '&', 
        '∨': '|',
        '⊕': '^',
        '¬': '-',
        '→': '->', 
        '—>': '->',
        '⟷': '<->',
        '↔': ' <-> '
    }
    for k, v in replacements.items():
        fol_text = fol_text.replace(k, v)

    # Add dot: "all x (P(x))" --> "all x. (P(x))"
    fol_text = re.sub(r'(all|exists)\s+([a-z0-9]+)\s*', r'\1 \2. ', fol_text)

    # Fix prover9 constants name (eg: "yuri" -> "c_yuri")
    words = re.findall(r'\b[a-z][a-zA-Z0-9_]*\b', fol_text)
    reserved_words = {'all', 'exists', 'u', 'v', 'w', 'x', 'y', 'z'}
    
    for w in set(words):
        if w not in reserved_words:
            fol_text = re.sub(fr'\b{w}\b', f'c_{w}', fol_text)
    return fol_text

In [82]:
file_path = 'folio_test.json'
with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

error_ids = set()
for d in data:
    fol_list = d["fol"].split('\n')
    for fol in fol_list:
        try:
            read_expr = Expression.fromstring(translate_fol(fol))
        except Exception as e:
            id = d["story_id"]
            print(f"{id}: {e}")
            error_ids.add(id)

print(sorted(error_ids))


[]


Fix files

In [73]:
import re

def fix_logic_errors(text):
    if not isinstance(text, str) or not text.strip(): return text

    # 2. XỬ LÝ LỖI HẰNG SỐ (Constants)
    # Sửa dấu chấm trong tên hằng số: c_mr_Andmrs_Smith -> c_mr_AndMrs_Smith
    text = re.sub(r'c_[a-zA-Z0-9\._]+', lambda m: m.group(0).replace('.', '_'), text)
    # Sửa khoảng trắng trong hằng số: c_xm c_bolt -> c_xm_bolt
    text = text.replace('c_xm c_bolt', 'c_xm_bolt')
    # Sửa dấu gạch ngang trong hằng số: c_l-2021 -> c_l_2021
    text = re.sub(r'([a-zA-Z0-9_]+)-([a-zA-Z0-9_]+)', r'\1_\2', text)

    # 3. XỬ LÝ LỖI LƯỢNG TỪ (Quantifiers)
    # Sửa lỗi exists x y.  -> exists x. exists y.
    text = re.sub(r'exists\s+c_([x-z])exists\.\s*([x-z])', r'exists \1. exists \2. ', text)
    # Sửa lỗi exists c_t. -> exists t.
    text = re.sub(r'exists\s+c_([a-z])(\s|\.)', r'exists \1 ', text)
    # Sửa lỗi dư dấu chấm sau lượng từ: exists x. -> exists x.
    text = text.replace('. .', '.')
    # Sửa lỗi thiếu biến: exists (Predicate... -> exists x. (Predicate...)
    text = re.sub(r'exists\s*\(', 'exists x. (', text)

    # 4. XỬ LÝ DẤU NGOẶC VÀ CẤU TRÚC VỊ TỪ
    # Sửa lỗi quên đóng ngoặc vị từ trước toán tử logic
    text = re.sub(r'([A-Z][a-zA-Z0-9_]*\([^)]+?)(?=\s*[&|^\-><])', r'\1)', text)
    
    # 5. DỌN DẸP CÁC KÝ TỰ DƯ THỪA
    # Xóa ghi chú "Never:" và phần sau nó
    if "Never:" in text:
        text = text.split("Never:")[0]
    # Xóa dấu chấm kết thúc câu (Parser không đọc dấu chấm văn bản)
    text = text.strip()
    if text.endswith('.'): text = text[:-1]
    # Xóa dấu phẩy thừa trong danh sách đối số
    text = text.replace(',)', ')')

    # 6. CÂN BẰNG NGOẶC THÔNG MINH
    # Xóa các dấu ngoặc đóng dư thừa đứng trước các toán tử chính hoặc ở cuối câu
    # Ví dụ: P(x)) -> Q(x) biến thành P(x) -> Q(x)
    for _ in range(3): # Chạy vài lần để xóa hết các lớp ngoặc dư
        text = re.sub(r'\)\)+(\s*(?:->|&|\||\^))', r')\1', text)
        
    open_p = text.count('(')
    close_p = text.count(')')
    
    if open_p > close_p:
        text += ')' * (open_p - close_p)
    elif close_p > open_p:
        # Xóa bớt dấu đóng ngoặc thừa ở CUỐI câu cho đến khi cân bằng
        while text.endswith(')') and text.count(')') > text.count('('):
            text = text[:-1]

    # Khử lỗi ngoặc kép bao quanh biến
    text = text.replace('((x)', '(x)').replace('((y)', '(y)')

    return text.strip()

def fix_file(file_path, delete_stories = []):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    data = [d for d in data if d["story_id"] not in delete_stories]
    for d in data:
        fol_list = d["fol"].split('\n')
        fixed_fol_list = [fix_logic_errors(s) for s in fol_list]
        d["fol"] = '\n'.join(fixed_fol_list)
    
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)


In [ ]:
fix_file('folio_train.json', [11, 103, 155, 341, 373, 376, 401, 409, 410, 411, 445, 447])
fix_file('folio_valid.json')
fix_file('folio_test.json', [380])

# Statistics

In [83]:
splits = ["train", "valid", "test"]
for spl in splits:
    with open(f"folio_{spl}.json", 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f"{spl}'s count:", len(data))

train's count: 955
valid's count: 100
test's count: 100
